Comenzaremos con la lectura de la base de datos provista en formato .parquet

In [ ]:
import polars as pl
from pathlib import Path

ROOT = Path('..')

In [2]:
datos = pl.read_parquet(ROOT/"data"/"raw"/"temporada1.parquet")

Una vez leidos los datos vamos a trabajar nuestra variable respuesta 'description' la cual presenta categorías que indican el resultado del pitcheo.

In [3]:
list(datos["description"].unique())

['hit_into_play',
 'foul_pitchout',
 'missed_bunt',
 'blocked_ball',
 'pitchout',
 'foul_tip',
 'foul',
 'called_strike',
 'swinging_strike',
 'bunt_foul_tip',
 'foul_bunt',
 'swinging_strike_blocked',
 'hit_by_pitch',
 'ball']

Podemos observar que los lanzamientos derivan en 14 distintos desenlaces, agrupandose en 4 resultados posibles: un **strike** (malo para el bateador), una **bola** (malo para el lanzador), un **foul** o una **pelota puesta en juego**.

- Lanzamientos que resultan en "Strike"

`called_strike`: El lanzador tira la pelota dentro de la zona de strike, el bateador no hace swing (no intenta batear) y el árbitro canta el strike.

`swinging_strike`: El bateador hace swing (intenta pegarle a la pelota) pero falla por completo.

`swinging_strike_blocked`: El bateador hace swing, falla, pero la pelota picó en el suelo antes de que el receptor la atrapara (regla especial donde si el lanzamiento es el tercer strike y el receptor no logra bloquearlo ,es decir la pelota se le escapa o va muy lejos, el bateador tiene la oportunidad de correr a primera base y salvarse de ser eliminado).

`missed_bunt`: El bateador intenta poner el bate para hacer un toque, pero falla por completo.

- Lanzamientos que resultan en "Ball"

`ball`: El bateador no hizo swing y el lanzamiento resultó fuera de la zona de strike.

`blocked_ball`: Una bola que pica en el suelo antes de llegar al receptor y este logra bloquearla con el cuerpo para que no se le escape. Tampoco se realiza un swing.

`pitchout`: Un lanzamiento deliberadamente muy abierto y alto. El lanzador lo hace a propósito porque sospecha que un corredor base va a intentar robarse la siguiente base; así le da ventaja a su receptor para atraparla rápido y lanzar.


- Lanzamientos que resultan en "Foul"

`foul`: El bateador hizo swing, impacta la pelota y la misma cae fuera de los límites laterales permitidos del campo. Suma como strike si el bateador tiene 0 o 1 strike, si ya tiene 2 strikes el conteo queda igual y se repite el tiro.

`foul_tip`: El bateador hace el swing, llega a rozar la pelota con el bate y esta va directo al guante del receptor, quien la atrapa limpiamente (esto siempre cuenta como strike, incluso si es el tercero). Quizas correspondería en apartado de **strike** pero en el nombre tiene foul (charlar con fran y agus).

`foul_bunt`: El bateador intenta hacer un toque, pero la pelota rebota hacia zona de foul (A diferencia de un foul normal, si intentas hacer un bunt con 2 strikes y se va foul, es out automático). Igual que foul_tip charlar

`foul_bunt_tip`: El bateador intenta un toque, la pelota roza levemente el bate y el receptor la atrapa directamente. Igual que anteriores.

`foul_pitchout`: El lanzador tiró un pitchout , pero el bateador saltó o extendió los brazos e intentó batearla, haciendo que se fuera al territorio de foul. Es una jugada extremadamente atípica.

- Lanzamientos que ponen en juego la pelota o que derivan en situaciones especiales

`hit_into_play`: El bateador le pega a la pelota y cae dentro del terreno válido. Esto desencadena los eventos principales del béisbol: puede terminar en un hit (el bateador llega a base) o en un out (la defensa atrapa la pelota en el aire o tira a la base).

`hit_by_pitch`: El lanzador golpea accidentalmente el cuerpo del bateador con la pelota. El bateador avanza automáticamente a primera base.


## Fuentes Oficiales

OFFICIAL BASEBALL RULES
2023 Edition
https://img.mlbstatic.com/mlb-images/image/upload/mlb/wqn5ah4c3qtivwx3jatm.pdf

Mapeo swing → variable binaria
La fuente de dominio más rigurosa que justifica este mapeo es el libro académico Analyzing Baseball Data with R, 3rd Edition (Baumer, Albert & Marchi), que en su capítulo sobre catcher framing trabaja exactamente con esta variable. El libro realiza el siguiente recodificado explícito de description: clasifica como "swing" las categorías swinging_strike, swinging_strike_blocked, foul, foul_bunt, foul_tip, hit_into_play y missed_bunt; y como "ball" o "called_strike" (no-swing) a ball, blocked_ball, pitchout y hit_by_pitch.
Las dos categorías restantes de tu dataset (bunt_foul_tip y foul_pitchout) no aparecen en ese ejemplo, pero su clasificación se desprende directamente del dominio: ambas involucran una acción deliberada del bateador.
https://beanumber.github.io/abdwr3e/07-framing.html

También en este artículo trabajan con este tipo de variable: Nestico, T. (2024). Creating the Perfect Pitching Summary
https://medium.com/@thomasjamesnestico/creating-the-perfect-pitching-summary-7b8a981ef0c5

Nota sobre `foul_pitchout`: es el caso más ambiguo. Técnicamente implica que el bateador hizo swing, pero como el pitchout está diseñado para ser imbateable y el evento es extremadamente raro (el lanzamiento es super alto y alejado), la mayoría de los marcos analíticos lo tratan como no-swing junto con pitchout. En ninguno de los 2 artículos mencionados anteriormente lo toman en cuenta como swing y lo veo con mucha lógica ya que no es un lanzamiento común que un bateador podría intentar golpear, es un caso super atípico que puede afectar las predicciones del modelo.